In [1]:
import ast
import os
import pandas as pd

def extract_genus_name(x):
    try:
        genus_dict = ast.literal_eval(x)
        return genus_dict.get('name', None)
    except:
        return None

os.chdir('/active-data/analysis_results/chr_pla')
all_data = pd.read_csv('genome_chr-pla_statistics.csv')
all_data['genus_clean'] = all_data['genus'].apply(extract_genus_name)
counts = all_data['genus_clean'].value_counts()
keep_genus = counts[counts >= 400].index.to_list()
print(keep_genus)

['Escherichia', 'Klebsiella', 'Staphylococcus', 'Pseudomonas', 'Bacillus', 'Salmonella', 'Streptococcus', 'Streptomyces', 'Acinetobacter', 'Enterococcus', 'Bordetella', 'Enterobacter', 'Xanthomonas', 'Campylobacter', 'Vibrio', 'Mycobacterium', 'Corynebacterium', 'Burkholderia', 'Listeria', 'Citrobacter', 'Helicobacter']


In [2]:
from tqdm import tqdm

for genus_name in keep_genus:
    base_folder = f'/active-data/analysis_results/chr_pla/genus'
    folder = f'{base_folder}/statistics_records/{genus_name}'
    os.chdir(folder)
    replicon_data = pd.read_csv('replicon-plasmid_fraction-self_bitscore_statistics.csv')
    with tqdm(total = len(replicon_data), desc=f'{genus_name}({len(replicon_data)})', leave=True, ncols=100, unit='B', unit_scale=True) as pbar:
        for index, row in replicon_data.iterrows():
            acc_n = row['accession'].split('-')[0]
            contig = row['accession'].split('-')[1]
            kmer_dir = f'/active-data/genomes/bacteria_complete_annotationRefSeq-20250807/kmer/{acc_n}/{contig}.txt'
            with open(kmer_dir, 'r', encoding='utf-8', errors='replace') as f:
                line = f.readline()
                line = line.replace('\x00', '')
            kmer_dict = ast.literal_eval(line)
            GC_content = (kmer_dict['1-mer']['G'] + kmer_dict['1-mer']['C'])/sum(kmer_dict['1-mer'].values())
            replicon_data.loc[index, 'GC_content'] = GC_content
            pbar.update(1)
    replicon_data.to_csv('replicon-plasmid_fraction-self_bitscore_statistics.csv', index=False)

Escherichia(15436): 100%|████████████████████████████████████████| 15.4k/15.4k [01:17<00:00, 199B/s]
Klebsiella(14975): 100%|█████████████████████████████████████████| 15.0k/15.0k [01:15<00:00, 199B/s]
Staphylococcus(5078): 100%|██████████████████████████████████████| 5.08k/5.08k [00:25<00:00, 203B/s]
Pseudomonas(3141): 100%|█████████████████████████████████████████| 3.14k/3.14k [00:15<00:00, 197B/s]
Bacillus(3992): 100%|████████████████████████████████████████████| 3.99k/3.99k [00:20<00:00, 198B/s]
Salmonella(4324): 100%|██████████████████████████████████████████| 4.32k/4.32k [00:21<00:00, 200B/s]
Streptococcus(1777): 100%|███████████████████████████████████████| 1.78k/1.78k [00:09<00:00, 197B/s]
Streptomyces(2461): 100%|████████████████████████████████████████| 2.46k/2.46k [00:12<00:00, 198B/s]
Acinetobacter(3743): 100%|███████████████████████████████████████| 3.74k/3.74k [00:18<00:00, 198B/s]
Enterococcus(3287): 100%|████████████████████████████████████████| 3.29k/3.29k [00:16<00:00